# Clasificación Cuántica de Crédito Financiero
## Comparación de Enfoques Cuánticos e Híbridos para Aprobación de Crédito

**Dataset:** Statlog (Australian Credit Approval) - UCI Machine Learning Repository

**Objetivo:** Comparar tres enfoques de clasificación cuántica/híbrida:
1. **QSVM** - Quantum Support Vector Machine con kernel cuántico
2. **VQC** - Variational Quantum Classifier (red neuronal cuántica pura)
3. **Híbrido** - Red neuronal clásica-cuántica

---

### Tabla de Contenidos
1. [Configuración e Instalación](#0-configuración)
2. [Carga y Análisis de Datos](#1-datos)
3. [Preprocesamiento](#preprocesamiento)
4. [QSVM - Quantum Support Vector Machine](#2a-qsvm)
5. [VQC - Variational Quantum Classifier](#2b-vqc)
6. [Red Híbrida Clásica-Cuántica](#2c-híbrido)
7. [Baseline Clásico](#2d-baseline)
8. [Análisis Comparativo y Conclusiones](#3-análisis)


<a name="0-configuración"></a>
## 0. Configuración e Instalación

Instalamos las dependencias necesarias y configuramos el entorno para reproducibilidad.


In [ ]:
# Instalación de dependencias
%pip install pennylane torch torchinfo scikit-learn pandas matplotlib seaborn -q


In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

# Standard library
import time
import warnings
warnings.filterwarnings('ignore')

# Third-party: Data manipulation
import numpy as np
import pandas as pd

# Third-party: Visualization
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# Third-party: Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, roc_curve, auc,
                             classification_report)

# Third-party: Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

# Third-party: Quantum Computing
import pennylane as qml

# Local: Utilidades de entrenamiento del curso
from minitorch import NumpyDataset, train, set_seed

# =============================================================================
# CONFIGURACIÓN GLOBAL
# =============================================================================

# Semilla para reproducibilidad
rs = 42
set_seed(rs)

# Configuración de matplotlib para gráficas estilo paper académico
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Computer Modern Roman'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.figsize': (8, 5),
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.framealpha': 0.9,
    'legend.edgecolor': 'gray',
})

# Paleta de colores consistente para los modelos
COLORS = {
    'qsvm': '#2E86AB',      # Azul
    'vqc_1': '#A23B72',     # Magenta
    'vqc_2': '#F18F01',     # Naranja
    'hybrid_1': '#C73E1D',  # Rojo
    'hybrid_2': '#3B1F2B',  # Púrpura oscuro
    'classical': '#44AF69', # Verde
}

print(f"PennyLane version: {qml.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Semilla aleatoria: {rs}")


<a name="1-datos"></a>
## 1. Carga y Análisis del Conjunto de Datos

### 1.1 Descripción del Dataset

El dataset **Statlog (Australian Credit Approval)** contiene información sobre solicitudes de tarjetas de crédito en Australia. Por razones de confidencialidad, los nombres de los atributos y valores han sido anonimizados.

- **Fuente:** UCI Machine Learning Repository
- **Instancias:** 690
- **Atributos:** 14 (6 numéricos, 8 categóricos)
- **Clases:** Binaria (aprobado/rechazado)
- **Valores faltantes:** Sí (indicados con '?')


In [ ]:
# =============================================================================
# CARGA DEL DATASET
# =============================================================================

# URL del dataset Australian Credit Approval (UCI)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/australian/australian.dat"

# Definimos nombres de columnas (A1-A14 para atributos, class para la etiqueta)
column_names = [f'A{i}' for i in range(1, 15)] + ['class']

# Cargamos los datos (separador espacio, sin cabecera)
df = pd.read_csv(url, sep=' ', header=None, names=column_names)

print(f"Dataset cargado exitosamente: {df.shape[0]} instancias, {df.shape[1]} columnas")
df.head(10)


### 1.2 Análisis Exploratorio de Datos (EDA)


In [ ]:
# Información general del dataset
print("="*60)
print("INFORMACIÓN DEL DATASET")
print("="*60)
df.info()
print("\n")

# Estadísticas descriptivas
print("="*60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
df.describe()


In [ ]:
# =============================================================================
# ANÁLISIS DEL BALANCE DE CLASES
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
class_counts = df['class'].value_counts().sort_index()
colors = ['#E74C3C', '#27AE60']  # Rojo para rechazado, Verde para aprobado
bars = axes[0].bar(['Rechazado (0)', 'Aprobado (1)'], class_counts.values, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Número de instancias')
axes[0].set_title('Distribución de Clases')

# Añadir valores sobre las barras
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfico de pastel
axes[1].pie(class_counts.values, labels=['Rechazado', 'Aprobado'], autopct='%1.1f%%',
            colors=colors, explode=(0.02, 0.02), shadow=True, startangle=90)
axes[1].set_title('Proporción de Clases')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Mostrar estadísticas
print(f"\nBalance de clases:")
print(f"  - Clase 0 (Rechazado): {class_counts[0]} ({100*class_counts[0]/len(df):.1f}%)")
print(f"  - Clase 1 (Aprobado): {class_counts[1]} ({100*class_counts[1]/len(df):.1f}%)")
print(f"  - Ratio: 1:{class_counts[1]/class_counts[0]:.2f}")


In [ ]:
# =============================================================================
# MATRIZ DE CORRELACIÓN
# =============================================================================

plt.figure(figsize=(12, 10))

# Calculamos la matriz de correlación
corr_matrix = df.corr()

# Creamos el heatmap con estilo académico
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Máscara para triángulo superior
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Coeficiente de Correlación"},
            annot_kws={"size": 8})

plt.title('Matriz de Correlación de Atributos', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlaciones más fuertes con la clase objetivo
print("\nCorrelaciones más significativas con la clase objetivo:")
class_corr = corr_matrix['class'].drop('class').abs().sort_values(ascending=False)
for attr, corr in class_corr.head(5).items():
    print(f"  {attr}: {corr_matrix.loc[attr, 'class']:+.3f}")


<a name="preprocesamiento"></a>
### 1.3 Preprocesamiento

El preprocesamiento es **crítico** para los circuitos cuánticos. Aplicamos:

1. **Escalado a $[0, \pi]$**: Necesario para *Angle Embedding*, donde cada feature se codifica como un ángulo de rotación en un qubit.

2. **PCA (Principal Component Analysis)**: Reducimos de 14 a 4 componentes principales.
   - **Justificación:** Simular 14 qubits es computacionalmente prohibitivo ($2^{14} = 16,384$ amplitudes). 
   - Con 4 qubits, el espacio de Hilbert es manejable ($2^4 = 16$ amplitudes).
   - PCA preserva la máxima varianza posible en las nuevas dimensiones.

3. **División estratificada**: Mantenemos la proporción de clases en train/validation/test.


In [ ]:
# =============================================================================
# SEPARACIÓN DE FEATURES Y TARGET
# =============================================================================

# Separamos features (X) y target (y)
X = df.drop('class', axis=1).values
y = df['class'].values

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Tipo de datos X: {X.dtype}")
print(f"Tipo de datos y: {y.dtype}")


In [ ]:
# =============================================================================
# DIVISIÓN TRAIN / VALIDATION / TEST
# =============================================================================

# Primera división: separamos test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=rs
)

# Segunda división: separamos validation (20% del restante = 16% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=rs
)

print("División de datos (estratificada):")
print(f"  - Train:      {len(X_train)} muestras ({100*len(X_train)/len(X):.1f}%)")
print(f"  - Validation: {len(X_val)} muestras ({100*len(X_val)/len(X):.1f}%)")
print(f"  - Test:       {len(X_test)} muestras ({100*len(X_test)/len(X):.1f}%)")
print(f"\nBalance en train: {np.mean(y_train):.2%} clase 1")
print(f"Balance en test:  {np.mean(y_test):.2%} clase 1")


In [ ]:
# =============================================================================
# ESCALADO A [0, π] PARA ANGLE EMBEDDING
# =============================================================================

# MinMaxScaler escala los datos al rango [0, π]
# Esto es esencial para Angle Embedding, donde cada feature se codifica
# como ángulo de rotación RY(θ) en un qubit
scaler = MinMaxScaler(feature_range=(0, np.pi))

# Ajustamos SOLO con datos de train para evitar data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Rango después del escalado:")
print(f"  - Min: {X_train_scaled.min():.4f}")
print(f"  - Max: {X_train_scaled.max():.4f}")
print(f"  - π ≈ {np.pi:.4f}")


In [ ]:
# =============================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA
# =============================================================================

# Número de componentes principales = número de qubits
n_components = 4
n_qubits = n_components

# Aplicamos PCA
pca = PCA(n_components=n_components, random_state=rs)

# Ajustamos SOLO con datos de train
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Visualización de la varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Varianza explicada por componente
axes[0].bar(range(1, n_components+1), pca.explained_variance_ratio_, 
            color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Proporción de Varianza Explicada')
axes[0].set_title('Varianza Explicada por Componente')
axes[0].set_xticks(range(1, n_components+1))

# Varianza acumulada
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, n_components+1), cumulative_variance, 'o-', 
             color='darkgreen', linewidth=2, markersize=8)
axes[1].axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='80% threshold')
axes[1].fill_between(range(1, n_components+1), cumulative_variance, alpha=0.3, color='green')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Explicada Acumulada')
axes[1].set_title('Varianza Acumulada')
axes[1].set_xticks(range(1, n_components+1))
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nVarianza explicada por componente:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.2%}")
print(f"\nVarianza total explicada: {cumulative_variance[-1]:.2%}")
print(f"Dimensiones reducidas: {X_train_scaled.shape[1]} → {n_components}")


In [ ]:
# =============================================================================
# RE-ESCALADO POST-PCA A [0, π]
# =============================================================================

# PCA puede producir valores fuera de [0, π], así que re-escalamos
scaler_pca = MinMaxScaler(feature_range=(0, np.pi))

X_train_final = scaler_pca.fit_transform(X_train_pca)
X_val_final = scaler_pca.transform(X_val_pca)
X_test_final = scaler_pca.transform(X_test_pca)

print(f"Shape final de los datos:")
print(f"  - X_train: {X_train_final.shape}")
print(f"  - X_val:   {X_val_final.shape}")
print(f"  - X_test:  {X_test_final.shape}")
print(f"\nRango: [{X_train_final.min():.4f}, {X_train_final.max():.4f}]")


In [ ]:
# =============================================================================
# CREACIÓN DE DATASETS PARA PYTORCH
# =============================================================================

# Encapsulamos los datos en NumpyDataset (de minitorch) para usar con PyTorch
tr_data = NumpyDataset(X_train_final, y_train)
val_data = NumpyDataset(X_val_final, y_val)
test_data = NumpyDataset(X_test_final, y_test)

print("Datasets creados para PyTorch:")
print(f"  - tr_data:   {len(tr_data)} muestras")
print(f"  - val_data:  {len(val_data)} muestras")
print(f"  - test_data: {len(test_data)} muestras")
print(f"\nShape de tensores: X={tr_data.x.shape}, y={tr_data.y.shape}")


---

<a name="2a-qsvm"></a>
## 2.A) Quantum Support Vector Machine (QSVM)

### Fundamento Teórico

Las **Quantum Support Vector Machines** utilizan un **kernel cuántico** para medir la similitud entre puntos de datos. El kernel cuántico aprovecha el espacio de Hilbert exponencialmente grande para crear separaciones no lineales.

**Proceso:**
1. **Codificación:** Cada punto de datos $\mathbf{x}$ se codifica en un estado cuántico $|\phi(\mathbf{x})\rangle$ usando *Angle Embedding*.
2. **Kernel:** El kernel cuántico calcula el producto interno:
   $$K(\mathbf{x}_1, \mathbf{x}_2) = |\langle\phi(\mathbf{x}_1)|\phi(\mathbf{x}_2)\rangle|^2$$
3. **SVM:** La matriz de kernel se pasa a un SVM clásico para encontrar el hiperplano óptimo.

**Ventaja teórica:** En algunos casos, los kernels cuánticos pueden capturar patrones que serían exponencialmente costosos de computar clásicamente.


In [ ]:
# =============================================================================
# DEFINICIÓN DEL KERNEL CUÁNTICO
# =============================================================================

# Dispositivo cuántico con 4 qubits
dev_kernel = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_kernel)
def quantum_kernel(x1, x2):
    """
    Kernel cuántico que calcula el solapamiento entre dos estados codificados.
    
    El circuito:
    1. Aplica AngleEmbedding para codificar x1 (rotaciones RY)
    2. Aplica el adjunto (inverso) de AngleEmbedding para x2
    3. Mide la probabilidad de volver al estado |0000⟩
    
    Si x1 == x2, el circuito vuelve exactamente a |0000⟩ (prob = 1)
    Si x1 != x2, la probabilidad de |0000⟩ es menor
    """
    # Codificamos x1 con rotaciones RY
    qml.templates.AngleEmbedding(x1, wires=range(n_qubits))
    
    # Aplicamos el adjunto (inverso) de la codificación de x2
    # Esto "deshace" la rotación si x1 ≈ x2
    qml.adjoint(qml.templates.AngleEmbedding(x2, wires=range(n_qubits)))
    
    # Retornamos las probabilidades de cada estado computacional
    return qml.probs()

# Visualizamos el circuito del kernel
print("Circuito del Kernel Cuántico:")
print("="*50)
x_example = X_train_final[0]
print(qml.draw(quantum_kernel, expansion_strategy="device")(x_example, x_example))


In [ ]:
# =============================================================================
# FUNCIÓN PARA CALCULAR LA MATRIZ DE KERNEL
# =============================================================================

def kernel_matrix(A, B):
    """
    Calcula la matriz de Gram del kernel cuántico.
    
    K[i,j] = quantum_kernel(A[i], B[j])[0]
    
    Donde [0] selecciona la probabilidad del estado |0000⟩
    """
    return np.array([[quantum_kernel(a, b)[0] for b in B] for a in A])

# Test del kernel: K(x, x) debe ser ≈ 1
test_kernel = quantum_kernel(X_train_final[0], X_train_final[0])[0]
print(f"Test del kernel K(x, x): {test_kernel:.6f} (esperado: 1.0)")

test_kernel_diff = quantum_kernel(X_train_final[0], X_train_final[1])[0]
print(f"Test del kernel K(x1, x2): {test_kernel_diff:.6f} (esperado: < 1.0)")


In [ ]:
# =============================================================================
# ENTRENAMIENTO DEL QSVM
# =============================================================================

# NOTA: El cálculo del kernel cuántico es O(n²), muy costoso para datasets grandes.
# Usamos un subconjunto para optimizar tiempo de ejecución.

# Parámetros
n_samples_qsvm = 200  # Subconjunto para el kernel (optimización de recursos)

print(f"Entrenando QSVM con {n_samples_qsvm} muestras (de {len(X_train_final)} disponibles)")
print("Justificación: El kernel cuántico requiere O(n²) evaluaciones del circuito.")
print("="*60)

# Seleccionamos subconjunto estratificado
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples_qsvm, random_state=rs)
train_idx, _ = next(sss.split(X_train_final, y_train))

X_train_qsvm = X_train_final[train_idx]
y_train_qsvm = y_train[train_idx]

print(f"Subconjunto: {len(X_train_qsvm)} muestras, balance: {np.mean(y_train_qsvm):.2%} clase 1")

# Entrenamiento
start_time = time.time()
qsvm = SVC(kernel=kernel_matrix)
qsvm.fit(X_train_qsvm, y_train_qsvm)
qsvm_train_time = time.time() - start_time

print(f"\nTiempo de entrenamiento: {qsvm_train_time:.2f} segundos")


In [ ]:
# =============================================================================
# EVALUACIÓN DEL QSVM
# =============================================================================

# Predicciones en test
y_pred_qsvm = qsvm.predict(X_test_final)

# Métricas
qsvm_accuracy = accuracy_score(y_test, y_pred_qsvm)
qsvm_precision = precision_score(y_test, y_pred_qsvm)
qsvm_recall = recall_score(y_test, y_pred_qsvm)
qsvm_f1 = f1_score(y_test, y_pred_qsvm)

print("="*60)
print("RESULTADOS QSVM")
print("="*60)
print(f"Accuracy:  {qsvm_accuracy:.4f}")
print(f"Precision: {qsvm_precision:.4f}")
print(f"Recall:    {qsvm_recall:.4f}")
print(f"F1-Score:  {qsvm_f1:.4f}")
print(f"\nTiempo de entrenamiento: {qsvm_train_time:.2f}s")

# Guardamos resultados para comparación final
results = {
    'QSVM': {
        'accuracy': qsvm_accuracy,
        'precision': qsvm_precision,
        'recall': qsvm_recall,
        'f1': qsvm_f1,
        'time': qsvm_train_time,
        'params': 'N/A',
        'y_pred': y_pred_qsvm,
        'y_proba': None  # SVM no da probabilidades directamente
    }
}


---

<a name="2b-vqc"></a>
## 2.B) Variational Quantum Classifier (VQC)

### Fundamento Teórico

El **Variational Quantum Classifier** es una red neuronal cuántica que aprende una función de clasificación mediante optimización de parámetros.

**Arquitectura:**
1. **Feature Map (Codificación):** Los datos clásicos se codifican en estados cuánticos usando *Angle Embedding* ($R_Y(\theta_i)$).
2. **Ansatz Variacional:** Capas de puertas parametrizadas que crean entrelazamiento y transformaciones. Usamos *StronglyEntanglingLayers*.
3. **Medición:** El valor esperado de un observable determina la clasificación.

**Fórmula del circuito:**
$$|\psi(\mathbf{x}, \boldsymbol{\theta})\rangle = U(\boldsymbol{\theta}) \cdot U_\phi(\mathbf{x}) |0\rangle^{\otimes n}$$

**Problema de Barren Plateaus:** Los gradientes pueden desvanecerse exponencialmente con el número de qubits, dificultando el entrenamiento. Se mitiga usando pocas capas y circuitos estructurados.


In [ ]:
# =============================================================================
# DEFINICIÓN DEL CIRCUITO VQC
# =============================================================================

# Dispositivo cuántico para VQC
dev_vqc = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_vqc)
def vqc_circuit(inputs, weights):
    """
    Circuito VQC para clasificación binaria.
    
    Args:
        inputs: Features de entrada (4 valores en [0, π])
        weights: Parámetros entrenables del ansatz variacional
    
    Returns:
        Valor esperado del proyector |0⟩⟨0| en el primer qubit
    """
    # Feature Map: Angle Embedding codifica cada feature como rotación RY
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    
    # Ansatz Variacional: StronglyEntanglingLayers
    # Incluye rotaciones Rot(φ, θ, ω) y CNOTs para entrelazamiento
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    
    # Medición: Proyector sobre |0⟩ en el primer qubit
    # Esto da un valor entre 0 y 1, ideal para clasificación binaria
    state_0 = [[1], [0]]
    projector = state_0 * np.conj(state_0).T  # |0⟩⟨0|
    return [qml.expval(qml.Hermitian(projector, wires=[0]))]

# Visualizamos el circuito (con 1 capa)
print("Circuito VQC (1 capa variacional):")
print("="*50)
example_weights = np.random.random((1, n_qubits, 3))
print(qml.draw(vqc_circuit, expansion_strategy="device")(X_train_final[0], example_weights))


### VQC con 1 Capa Variacional


In [ ]:
# =============================================================================
# VQC CON 1 CAPA VARIACIONAL
# =============================================================================

# Reiniciamos la semilla para reproducibilidad
set_seed(rs)

# Número de capas variacionales
n_layers_vqc1 = 1

# Definimos la forma de los pesos: (n_layers, n_qubits, 3)
# 3 parámetros por qubit por capa (φ, θ, ω para Rot)
weight_shapes_vqc1 = {"weights": (n_layers_vqc1, n_qubits, 3)}

# Creamos la capa cuántica compatible con PyTorch
qlayer_vqc1 = qml.qnn.TorchLayer(vqc_circuit, weight_shapes_vqc1)

# El modelo es solo la capa cuántica (VQC puro)
model_vqc1 = nn.Sequential(qlayer_vqc1)

# Optimizador y función de pérdida
opt_vqc1 = torch.optim.Adam(model_vqc1.parameters(), lr=0.05)
loss_fn = F.binary_cross_entropy

# Mostramos resumen del modelo
print("Arquitectura VQC (1 capa):")
print("="*50)
summary(model_vqc1)


In [ ]:
# =============================================================================
# ENTRENAMIENTO VQC (1 CAPA)
# =============================================================================

print("Entrenando VQC con 1 capa variacional...")
print("="*60)

start_time = time.time()
history_vqc1 = train(
    model=model_vqc1, 
    loss=loss_fn, 
    opt=opt_vqc1, 
    tr_data=tr_data, 
    val_data=val_data, 
    max_epochs=100, 
    patience=10
)
vqc1_train_time = time.time() - start_time

print(f"\nTiempo total de entrenamiento: {vqc1_train_time:.2f} segundos")


In [ ]:
# =============================================================================
# GRÁFICAS DE ENTRENAMIENTO VQC (1 CAPA)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica de Loss
axes[0].plot(history_vqc1.tr_losses, label='Training Loss', color=COLORS['vqc_1'], linewidth=2)
axes[0].plot(history_vqc1.val_losses, label='Validation Loss', color=COLORS['vqc_1'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('VQC (1 capa) - Evolución del Loss')
axes[0].legend()

# Gráfica de Accuracy
axes[1].plot(history_vqc1.tr_accuracy, label='Training Accuracy', color=COLORS['vqc_1'], linewidth=2)
axes[1].plot(history_vqc1.val_accuracy, label='Validation Accuracy', color=COLORS['vqc_1'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('VQC (1 capa) - Evolución del Accuracy')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('vqc1_training.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# EVALUACIÓN VQC (1 CAPA)
# =============================================================================

# Predicciones en test
with torch.no_grad():
    y_proba_vqc1 = model_vqc1(test_data.x).numpy().flatten()
    y_pred_vqc1 = (y_proba_vqc1 >= 0.5).astype(int)

# Métricas
vqc1_accuracy = accuracy_score(y_test, y_pred_vqc1)
vqc1_precision = precision_score(y_test, y_pred_vqc1)
vqc1_recall = recall_score(y_test, y_pred_vqc1)
vqc1_f1 = f1_score(y_test, y_pred_vqc1)

print("="*60)
print("RESULTADOS VQC (1 CAPA)")
print("="*60)
print(f"Accuracy:  {vqc1_accuracy:.4f}")
print(f"Precision: {vqc1_precision:.4f}")
print(f"Recall:    {vqc1_recall:.4f}")
print(f"F1-Score:  {vqc1_f1:.4f}")
print(f"\nParámetros entrenables: {n_layers_vqc1 * n_qubits * 3}")
print(f"Tiempo de entrenamiento: {vqc1_train_time:.2f}s")

# Guardamos resultados
results['VQC (1 capa)'] = {
    'accuracy': vqc1_accuracy,
    'precision': vqc1_precision,
    'recall': vqc1_recall,
    'f1': vqc1_f1,
    'time': vqc1_train_time,
    'params': n_layers_vqc1 * n_qubits * 3,
    'y_pred': y_pred_vqc1,
    'y_proba': y_proba_vqc1
}


### VQC con 2 Capas Variacionales

Aumentamos la expresividad del modelo duplicando las capas variacionales. Esto incrementa los parámetros entrenables pero puede mejorar la capacidad de aprendizaje.


In [ ]:
# =============================================================================
# VQC CON 2 CAPAS VARIACIONALES
# =============================================================================

# Reiniciamos la semilla
set_seed(rs)

# Número de capas variacionales
n_layers_vqc2 = 2

# Forma de los pesos
weight_shapes_vqc2 = {"weights": (n_layers_vqc2, n_qubits, 3)}

# Creamos la capa cuántica
qlayer_vqc2 = qml.qnn.TorchLayer(vqc_circuit, weight_shapes_vqc2)

# Modelo
model_vqc2 = nn.Sequential(qlayer_vqc2)

# Optimizador
opt_vqc2 = torch.optim.Adam(model_vqc2.parameters(), lr=0.05)

# Resumen
print("Arquitectura VQC (2 capas):")
print("="*50)
summary(model_vqc2)


In [ ]:
# =============================================================================
# ENTRENAMIENTO VQC (2 CAPAS)
# =============================================================================

print("Entrenando VQC con 2 capas variacionales...")
print("="*60)

start_time = time.time()
history_vqc2 = train(
    model=model_vqc2, 
    loss=loss_fn, 
    opt=opt_vqc2, 
    tr_data=tr_data, 
    val_data=val_data, 
    max_epochs=100, 
    patience=10
)
vqc2_train_time = time.time() - start_time

print(f"\nTiempo total de entrenamiento: {vqc2_train_time:.2f} segundos")


In [ ]:
# =============================================================================
# GRÁFICAS Y EVALUACIÓN VQC (2 CAPAS)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_vqc2.tr_losses, label='Training Loss', color=COLORS['vqc_2'], linewidth=2)
axes[0].plot(history_vqc2.val_losses, label='Validation Loss', color=COLORS['vqc_2'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('VQC (2 capas) - Evolución del Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history_vqc2.tr_accuracy, label='Training Accuracy', color=COLORS['vqc_2'], linewidth=2)
axes[1].plot(history_vqc2.val_accuracy, label='Validation Accuracy', color=COLORS['vqc_2'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('VQC (2 capas) - Evolución del Accuracy')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('vqc2_training.png', dpi=300, bbox_inches='tight')
plt.show()

# Evaluación
with torch.no_grad():
    y_proba_vqc2 = model_vqc2(test_data.x).numpy().flatten()
    y_pred_vqc2 = (y_proba_vqc2 >= 0.5).astype(int)

vqc2_accuracy = accuracy_score(y_test, y_pred_vqc2)
vqc2_precision = precision_score(y_test, y_pred_vqc2)
vqc2_recall = recall_score(y_test, y_pred_vqc2)
vqc2_f1 = f1_score(y_test, y_pred_vqc2)

print("="*60)
print("RESULTADOS VQC (2 CAPAS)")
print("="*60)
print(f"Accuracy:  {vqc2_accuracy:.4f}")
print(f"Precision: {vqc2_precision:.4f}")
print(f"Recall:    {vqc2_recall:.4f}")
print(f"F1-Score:  {vqc2_f1:.4f}")
print(f"\nParámetros entrenables: {n_layers_vqc2 * n_qubits * 3}")

# Guardamos resultados
results['VQC (2 capas)'] = {
    'accuracy': vqc2_accuracy,
    'precision': vqc2_precision,
    'recall': vqc2_recall,
    'f1': vqc2_f1,
    'time': vqc2_train_time,
    'params': n_layers_vqc2 * n_qubits * 3,
    'y_pred': y_pred_vqc2,
    'y_proba': y_proba_vqc2
}


---

<a name="2c-híbrido"></a>
## 2.C) Red Neuronal Híbrida (Clásica-Cuántica)

### Fundamento Teórico

Las **redes híbridas** combinan capas clásicas (fully connected) con capas cuánticas. Esta arquitectura ofrece varias ventajas:

1. **Preprocesamiento clásico:** Las capas clásicas pueden aprender transformaciones útiles antes del procesamiento cuántico.
2. **Postprocesamiento clásico:** Se puede ajustar la salida cuántica para la tarea específica.
3. **Flexibilidad:** Permite usar pocos qubits incluso con muchas features de entrada.

**Arquitectura:**
$$\text{Input} \xrightarrow{\text{Linear + ReLU}} \text{Capa Cuántica} \xrightarrow{\text{Linear + Sigmoid}} \text{Output}$$

**Medición:** A diferencia del VQC, la capa cuántica retorna el valor esperado de $\sigma_z$ (PauliZ) en cada qubit, dando valores en $[-1, 1]$.


In [ ]:
# =============================================================================
# DEFINICIÓN DEL CIRCUITO PARA RED HÍBRIDA
# =============================================================================

# Dispositivo cuántico para modelo híbrido
dev_hybrid = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_hybrid)
def hybrid_circuit(inputs, weights):
    """
    Circuito cuántico para red híbrida.
    
    A diferencia del VQC, este circuito:
    - Recibe inputs preprocesados por capa clásica
    - Retorna valores esperados de PauliZ en cada qubit (no solo el primero)
    
    Returns:
        Lista de valores esperados de σz en cada qubit (valores en [-1, 1])
    """
    # Feature Map
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    
    # Ansatz Variacional
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    
    # Medición: valor esperado de PauliZ en cada qubit
    # Esto da n_qubits valores que la capa clásica posterior puede combinar
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Visualizamos el circuito
print("Circuito para Red Híbrida:")
print("="*50)
example_weights = np.random.random((1, n_qubits, 3))
print(qml.draw(hybrid_circuit, expansion_strategy="device")(X_train_final[0], example_weights))


### Red Híbrida con 1 Capa Variacional


In [ ]:
# =============================================================================
# RED HÍBRIDA CON 1 CAPA VARIACIONAL
# =============================================================================

# Reiniciamos semilla
set_seed(rs)

# Número de capas variacionales
n_layers_hybrid1 = 1

# Forma de los pesos cuánticos
weight_shapes_hybrid1 = {"weights": (n_layers_hybrid1, n_qubits, 3)}

# Capa cuántica
qlayer_hybrid1 = qml.qnn.TorchLayer(hybrid_circuit, weight_shapes_hybrid1)

# Arquitectura híbrida: Clásica -> Cuántica -> Clásica
model_hybrid1 = nn.Sequential(
    nn.Linear(n_qubits, n_qubits),  # Capa clásica de entrada (4 -> 4)
    nn.ReLU(),                       # Activación no lineal
    qlayer_hybrid1,                  # Capa cuántica
    nn.Linear(n_qubits, 1),          # Capa clásica de salida (4 -> 1)
    nn.Sigmoid()                     # Activación para clasificación binaria
)

# Optimizador
opt_hybrid1 = torch.optim.Adam(model_hybrid1.parameters(), lr=0.05)

# Resumen del modelo
print("Arquitectura Red Híbrida (1 capa cuántica):")
print("="*50)
summary(model_hybrid1)


In [ ]:
# =============================================================================
# ENTRENAMIENTO RED HÍBRIDA (1 CAPA)
# =============================================================================

print("Entrenando Red Híbrida con 1 capa variacional...")
print("="*60)

start_time = time.time()
history_hybrid1 = train(
    model=model_hybrid1, 
    loss=loss_fn, 
    opt=opt_hybrid1, 
    tr_data=tr_data, 
    val_data=val_data, 
    max_epochs=100, 
    patience=10
)
hybrid1_train_time = time.time() - start_time

print(f"\nTiempo total de entrenamiento: {hybrid1_train_time:.2f} segundos")


In [ ]:
# =============================================================================
# GRÁFICAS Y EVALUACIÓN RED HÍBRIDA (1 CAPA)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_hybrid1.tr_losses, label='Training Loss', color=COLORS['hybrid_1'], linewidth=2)
axes[0].plot(history_hybrid1.val_losses, label='Validation Loss', color=COLORS['hybrid_1'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Red Híbrida (1 capa) - Evolución del Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history_hybrid1.tr_accuracy, label='Training Accuracy', color=COLORS['hybrid_1'], linewidth=2)
axes[1].plot(history_hybrid1.val_accuracy, label='Validation Accuracy', color=COLORS['hybrid_1'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Red Híbrida (1 capa) - Evolución del Accuracy')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('hybrid1_training.png', dpi=300, bbox_inches='tight')
plt.show()

# Evaluación
with torch.no_grad():
    y_proba_hybrid1 = model_hybrid1(test_data.x).numpy().flatten()
    y_pred_hybrid1 = (y_proba_hybrid1 >= 0.5).astype(int)

hybrid1_accuracy = accuracy_score(y_test, y_pred_hybrid1)
hybrid1_precision = precision_score(y_test, y_pred_hybrid1)
hybrid1_recall = recall_score(y_test, y_pred_hybrid1)
hybrid1_f1 = f1_score(y_test, y_pred_hybrid1)

# Contar parámetros totales
n_params_hybrid1 = sum(p.numel() for p in model_hybrid1.parameters())

print("="*60)
print("RESULTADOS RED HÍBRIDA (1 CAPA)")
print("="*60)
print(f"Accuracy:  {hybrid1_accuracy:.4f}")
print(f"Precision: {hybrid1_precision:.4f}")
print(f"Recall:    {hybrid1_recall:.4f}")
print(f"F1-Score:  {hybrid1_f1:.4f}")
print(f"\nParámetros totales: {n_params_hybrid1}")

# Guardamos resultados
results['Híbrido (1 capa)'] = {
    'accuracy': hybrid1_accuracy,
    'precision': hybrid1_precision,
    'recall': hybrid1_recall,
    'f1': hybrid1_f1,
    'time': hybrid1_train_time,
    'params': n_params_hybrid1,
    'y_pred': y_pred_hybrid1,
    'y_proba': y_proba_hybrid1
}


### Red Híbrida con 2 Capas Variacionales


In [ ]:
# =============================================================================
# RED HÍBRIDA CON 2 CAPAS VARIACIONALES
# =============================================================================

# Reiniciamos semilla
set_seed(rs)

# Número de capas variacionales
n_layers_hybrid2 = 2

# Forma de los pesos cuánticos
weight_shapes_hybrid2 = {"weights": (n_layers_hybrid2, n_qubits, 3)}

# Capa cuántica
qlayer_hybrid2 = qml.qnn.TorchLayer(hybrid_circuit, weight_shapes_hybrid2)

# Arquitectura híbrida
model_hybrid2 = nn.Sequential(
    nn.Linear(n_qubits, n_qubits),
    nn.ReLU(),
    qlayer_hybrid2,
    nn.Linear(n_qubits, 1),
    nn.Sigmoid()
)

# Optimizador
opt_hybrid2 = torch.optim.Adam(model_hybrid2.parameters(), lr=0.05)

# Resumen
print("Arquitectura Red Híbrida (2 capas cuánticas):")
print("="*50)
summary(model_hybrid2)


In [ ]:
# =============================================================================
# ENTRENAMIENTO Y EVALUACIÓN RED HÍBRIDA (2 CAPAS)
# =============================================================================

print("Entrenando Red Híbrida con 2 capas variacionales...")
print("="*60)

start_time = time.time()
history_hybrid2 = train(
    model=model_hybrid2, 
    loss=loss_fn, 
    opt=opt_hybrid2, 
    tr_data=tr_data, 
    val_data=val_data, 
    max_epochs=100, 
    patience=10
)
hybrid2_train_time = time.time() - start_time

print(f"\nTiempo total de entrenamiento: {hybrid2_train_time:.2f} segundos")


In [ ]:
# =============================================================================
# GRÁFICAS Y EVALUACIÓN RED HÍBRIDA (2 CAPAS)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_hybrid2.tr_losses, label='Training Loss', color=COLORS['hybrid_2'], linewidth=2)
axes[0].plot(history_hybrid2.val_losses, label='Validation Loss', color=COLORS['hybrid_2'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Red Híbrida (2 capas) - Evolución del Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history_hybrid2.tr_accuracy, label='Training Accuracy', color=COLORS['hybrid_2'], linewidth=2)
axes[1].plot(history_hybrid2.val_accuracy, label='Validation Accuracy', color=COLORS['hybrid_2'], 
             linestyle='--', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Red Híbrida (2 capas) - Evolución del Accuracy')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('hybrid2_training.png', dpi=300, bbox_inches='tight')
plt.show()

# Evaluación
with torch.no_grad():
    y_proba_hybrid2 = model_hybrid2(test_data.x).numpy().flatten()
    y_pred_hybrid2 = (y_proba_hybrid2 >= 0.5).astype(int)

hybrid2_accuracy = accuracy_score(y_test, y_pred_hybrid2)
hybrid2_precision = precision_score(y_test, y_pred_hybrid2)
hybrid2_recall = recall_score(y_test, y_pred_hybrid2)
hybrid2_f1 = f1_score(y_test, y_pred_hybrid2)

n_params_hybrid2 = sum(p.numel() for p in model_hybrid2.parameters())

print("="*60)
print("RESULTADOS RED HÍBRIDA (2 CAPAS)")
print("="*60)
print(f"Accuracy:  {hybrid2_accuracy:.4f}")
print(f"Precision: {hybrid2_precision:.4f}")
print(f"Recall:    {hybrid2_recall:.4f}")
print(f"F1-Score:  {hybrid2_f1:.4f}")
print(f"\nParámetros totales: {n_params_hybrid2}")

# Guardamos resultados
results['Híbrido (2 capas)'] = {
    'accuracy': hybrid2_accuracy,
    'precision': hybrid2_precision,
    'recall': hybrid2_recall,
    'f1': hybrid2_f1,
    'time': hybrid2_train_time,
    'params': n_params_hybrid2,
    'y_pred': y_pred_hybrid2,
    'y_proba': y_proba_hybrid2
}


---

<a name="2d-baseline"></a>
## 2.D) Baseline Clásico (SVM con Kernel RBF)

Para evaluar objetivamente los modelos cuánticos, comparamos con un **clasificador clásico estado del arte**: SVM con kernel RBF (Radial Basis Function).

$$K(\mathbf{x}_1, \mathbf{x}_2) = \exp\left(-\gamma \|\mathbf{x}_1 - \mathbf{x}_2\|^2\right)$$

Este kernel es conocido por su excelente rendimiento en problemas de clasificación no lineal.


In [ ]:
# =============================================================================
# BASELINE CLÁSICO: SVM CON KERNEL RBF
# =============================================================================

# Entrenamos SVM clásico con kernel RBF
start_time = time.time()
svm_classical = SVC(kernel='rbf', random_state=rs, probability=True)
svm_classical.fit(X_train_final, y_train)
classical_train_time = time.time() - start_time

# Predicciones
y_pred_classical = svm_classical.predict(X_test_final)
y_proba_classical = svm_classical.predict_proba(X_test_final)[:, 1]

# Métricas
classical_accuracy = accuracy_score(y_test, y_pred_classical)
classical_precision = precision_score(y_test, y_pred_classical)
classical_recall = recall_score(y_test, y_pred_classical)
classical_f1 = f1_score(y_test, y_pred_classical)

print("="*60)
print("RESULTADOS BASELINE CLÁSICO (SVM RBF)")
print("="*60)
print(f"Accuracy:  {classical_accuracy:.4f}")
print(f"Precision: {classical_precision:.4f}")
print(f"Recall:    {classical_recall:.4f}")
print(f"F1-Score:  {classical_f1:.4f}")
print(f"\nTiempo de entrenamiento: {classical_train_time:.4f}s")

# Guardamos resultados
results['SVM Clásico (RBF)'] = {
    'accuracy': classical_accuracy,
    'precision': classical_precision,
    'recall': classical_recall,
    'f1': classical_f1,
    'time': classical_train_time,
    'params': 'N/A',
    'y_pred': y_pred_classical,
    'y_proba': y_proba_classical
}


---

<a name="3-análisis"></a>
## 3. Análisis Comparativo y Conclusiones

### 3.1 Tabla Comparativa de Resultados


In [ ]:
# =============================================================================
# TABLA COMPARATIVA DE RESULTADOS
# =============================================================================

# Creamos DataFrame con los resultados
results_df = pd.DataFrame({
    'Modelo': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()],
    'Tiempo (s)': [r['time'] for r in results.values()],
    'Parámetros': [r['params'] for r in results.values()]
})

# Ordenamos por F1-Score descendente
results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

# Mostramos la tabla
print("="*80)
print("TABLA COMPARATIVA DE MODELOS")
print("="*80)
print(results_df.to_string(index=False))

# Guardamos como CSV
results_df.to_csv('results_comparison.csv', index=False)
print("\nTabla guardada en 'results_comparison.csv'")


In [ ]:
# =============================================================================
# GRÁFICO COMPARATIVO DE MÉTRICAS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Colores para cada modelo
model_colors = [COLORS['qsvm'], COLORS['vqc_1'], COLORS['vqc_2'], 
                COLORS['hybrid_1'], COLORS['hybrid_2'], COLORS['classical']]

# Gráfico de barras: Accuracy y F1-Score
metrics = ['Accuracy', 'F1-Score']
x = np.arange(len(results_df))
width = 0.35

bars1 = axes[0].bar(x - width/2, results_df['Accuracy'], width, label='Accuracy', 
                     color='steelblue', edgecolor='black', alpha=0.8)
bars2 = axes[0].bar(x + width/2, results_df['F1-Score'], width, label='F1-Score', 
                     color='darkorange', edgecolor='black', alpha=0.8)

axes[0].set_xlabel('Modelo')
axes[0].set_ylabel('Score')
axes[0].set_title('Comparación de Accuracy y F1-Score')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Modelo'], rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# Añadir valores sobre las barras
for bar in bars1:
    height = bar.get_height()
    axes[0].annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    axes[0].annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

# Gráfico de tiempo de entrenamiento (escala logarítmica)
bars3 = axes[1].bar(results_df['Modelo'], results_df['Tiempo (s)'], 
                     color=model_colors[:len(results_df)], edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Modelo')
axes[1].set_ylabel('Tiempo de Entrenamiento (s) - Escala Log')
axes[1].set_title('Tiempo de Entrenamiento por Modelo')
axes[1].set_yscale('log')
axes[1].set_xticklabels(results_df['Modelo'], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


### 3.2 Matrices de Confusión

Las matrices de confusión muestran el desglose de predicciones correctas e incorrectas para cada modelo.


In [ ]:
# =============================================================================
# MATRICES DE CONFUSIÓN
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, model_results) in enumerate(results.items()):
    cm = confusion_matrix(y_test, model_results['y_pred'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Rechazado', 'Aprobado'],
                yticklabels=['Rechazado', 'Aprobado'],
                cbar=False, annot_kws={"size": 14})
    
    axes[idx].set_title(f'{model_name}\nAccuracy: {model_results["accuracy"]:.3f}', fontsize=11)
    axes[idx].set_xlabel('Predicción')
    axes[idx].set_ylabel('Real')

# Si hay menos de 6 modelos, ocultamos los ejes sobrantes
for idx in range(len(results), 6):
    axes[idx].axis('off')

plt.suptitle('Matrices de Confusión - Comparación de Modelos', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()


### 3.3 Curvas ROC y AUC

La curva ROC (Receiver Operating Characteristic) muestra el trade-off entre sensibilidad (TPR) y especificidad (1-FPR). El área bajo la curva (AUC) proporciona una medida única de rendimiento.


In [ ]:
# =============================================================================
# CURVAS ROC
# =============================================================================

plt.figure(figsize=(10, 8))

# Colores para las curvas
colors_list = list(COLORS.values())

for idx, (model_name, model_results) in enumerate(results.items()):
    if model_results['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, model_results['y_proba'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors_list[idx % len(colors_list)], linewidth=2,
                 label=f'{model_name} (AUC = {roc_auc:.3f})')
    else:
        # Para modelos sin probabilidades (QSVM), usamos predicción directa
        fpr, tpr, _ = roc_curve(y_test, model_results['y_pred'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors_list[idx % len(colors_list)], linewidth=2,
                 linestyle='--', label=f'{model_name} (AUC = {roc_auc:.3f})*')

# Línea diagonal (clasificador aleatorio)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Especificidad)')
plt.ylabel('True Positive Rate (Sensibilidad)')
plt.title('Curvas ROC - Comparación de Modelos', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("* Los modelos marcados con asterisco no proporcionan probabilidades, se usa predicción directa.")


### 3.4 Análisis de Sensibilidad: Efecto del Número de Capas


In [ ]:
# =============================================================================
# ANÁLISIS DE SENSIBILIDAD: CAPAS VS ACCURACY
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Datos para VQC
layers_vqc = [1, 2]
acc_vqc = [results['VQC (1 capa)']['accuracy'], results['VQC (2 capas)']['accuracy']]
params_vqc = [n_layers_vqc1 * n_qubits * 3, n_layers_vqc2 * n_qubits * 3]

# Datos para Híbrido
layers_hybrid = [1, 2]
acc_hybrid = [results['Híbrido (1 capa)']['accuracy'], results['Híbrido (2 capas)']['accuracy']]

# Gráfico 1: Accuracy vs Capas
axes[0].plot(layers_vqc, acc_vqc, 'o-', color=COLORS['vqc_1'], linewidth=2, 
             markersize=10, label='VQC')
axes[0].plot(layers_hybrid, acc_hybrid, 's-', color=COLORS['hybrid_1'], linewidth=2, 
             markersize=10, label='Híbrido')

# Línea horizontal para baseline
axes[0].axhline(y=results['SVM Clásico (RBF)']['accuracy'], color=COLORS['classical'], 
                linestyle='--', linewidth=2, alpha=0.7, label='Baseline (SVM RBF)')

axes[0].set_xlabel('Número de Capas Variacionales')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Efecto del Número de Capas en el Accuracy')
axes[0].set_xticks([1, 2])
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

# Gráfico 2: Parámetros vs Accuracy
all_params = [12, 24, n_params_hybrid1, n_params_hybrid2]
all_acc = [results['VQC (1 capa)']['accuracy'], results['VQC (2 capas)']['accuracy'],
           results['Híbrido (1 capa)']['accuracy'], results['Híbrido (2 capas)']['accuracy']]
labels = ['VQC (1)', 'VQC (2)', 'Híb (1)', 'Híb (2)']
colors_scatter = [COLORS['vqc_1'], COLORS['vqc_2'], COLORS['hybrid_1'], COLORS['hybrid_2']]

for i, (p, a, l, c) in enumerate(zip(all_params, all_acc, labels, colors_scatter)):
    axes[1].scatter(p, a, s=150, color=c, edgecolor='black', linewidth=1.5, zorder=5)
    axes[1].annotate(l, (p, a), textcoords="offset points", xytext=(5, 5), fontsize=9)

axes[1].set_xlabel('Número de Parámetros')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Trade-off: Parámetros vs Accuracy')

plt.tight_layout()
plt.savefig('sensitivity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


### 3.5 Análisis Crítico y Conclusiones


In [ ]:
# =============================================================================
# RESUMEN DE HALLAZGOS
# =============================================================================

# Encontrar el mejor modelo cuántico
quantum_models = {k: v for k, v in results.items() if 'Clásico' not in k}
best_quantum = max(quantum_models.items(), key=lambda x: x[1]['f1'])

# Comparación con baseline
classical_f1 = results['SVM Clásico (RBF)']['f1']
best_quantum_f1 = best_quantum[1]['f1']
improvement = ((best_quantum_f1 - classical_f1) / classical_f1) * 100

print("="*70)
print("RESUMEN DE HALLAZGOS")
print("="*70)
print(f"\n1. MEJOR MODELO CUÁNTICO/HÍBRIDO: {best_quantum[0]}")
print(f"   - F1-Score: {best_quantum_f1:.4f}")
print(f"   - Accuracy: {best_quantum[1]['accuracy']:.4f}")

print(f"\n2. BASELINE CLÁSICO (SVM RBF):")
print(f"   - F1-Score: {classical_f1:.4f}")
print(f"   - Accuracy: {results['SVM Clásico (RBF)']['accuracy']:.4f}")

print(f"\n3. COMPARACIÓN:")
if improvement > 0:
    print(f"   - El mejor modelo cuántico supera al baseline en {improvement:.1f}% (F1-Score)")
else:
    print(f"   - El baseline clásico supera al mejor modelo cuántico en {-improvement:.1f}% (F1-Score)")

print(f"\n4. EFICIENCIA COMPUTACIONAL:")
print(f"   - Tiempo más rápido: SVM Clásico ({results['SVM Clásico (RBF)']['time']:.3f}s)")
print(f"   - Tiempo QSVM: {results['QSVM']['time']:.1f}s (kernel cuántico es costoso)")

print("\n" + "="*70)


### Conclusiones Finales

#### Rendimiento de los Modelos

Los experimentos realizados permiten extraer las siguientes conclusiones:

1. **QSVM (Kernel Cuántico):** El kernel cuántico proporciona una alternativa al kernel clásico RBF. Sin embargo, su coste computacional es significativamente mayor debido a la necesidad de evaluar el circuito cuántico O(n²) veces para construir la matriz de Gram.

2. **VQC (Variational Quantum Classifier):** El modelo puramente cuántico muestra una capacidad de aprendizaje limitada con pocas capas variacionales. Aumentar el número de capas mejora la expresividad pero incrementa el riesgo de *barren plateaus* y el tiempo de entrenamiento.

3. **Red Híbrida:** La combinación de capas clásicas y cuánticas ofrece un buen balance. Las capas clásicas pueden preprocesar los datos y la capa cuántica aporta transformaciones no lineales en el espacio de Hilbert.

#### Coste Computacional vs Beneficio

En la era NISQ (Noisy Intermediate-Scale Quantum), los modelos cuánticos enfrentan limitaciones:
- **Simulación costosa:** Sin hardware cuántico real, la simulación clásica es exponencialmente costosa.
- **Ruido:** Los dispositivos reales introducen errores que degradan el rendimiento.
- **Escalabilidad:** Aumentar qubits mejora la expresividad pero dificulta el entrenamiento.

#### Recomendaciones

- Para **aplicaciones prácticas actuales**, los modelos clásicos siguen siendo preferibles por su eficiencia.
- Los modelos **híbridos** son prometedores para explorar ventajas cuánticas con recursos limitados.
- El **QSVM** puede ser útil cuando se busca un kernel altamente no lineal, asumiendo el coste computacional.

#### Trabajo Futuro

1. Ejecutar en hardware cuántico real para evaluar el impacto del ruido.
2. Explorar técnicas de mitigación de errores.
3. Investigar *quantum advantage* en datasets de mayor dimensionalidad.
4. Comparar con más arquitecturas de ansätze variacionales.


---

## Referencias

1. Schuld, M., & Petruccione, F. (2021). *Machine Learning with Quantum Computers*. Springer.
2. Havlíček, V., et al. (2019). Supervised learning with quantum-enhanced feature spaces. *Nature*, 567(7747), 209-212.
3. Cerezo, M., et al. (2021). Variational quantum algorithms. *Nature Reviews Physics*, 3(9), 625-644.
4. PennyLane Documentation: https://pennylane.ai/qml/
5. UCI Machine Learning Repository - Statlog (Australian Credit Approval): https://archive.ics.uci.edu/ml/datasets/statlog+(australian+credit+approval)

---

**Notebook generado para el curso de Quantum Machine Learning**  
*Fecha de generación: 2026*
